# HCRIS Hospital Cost Report Explorer

Interactive tool to browse CMS Form 2552-10 hospital cost report data.
Select a **year**, **hospital**, and **worksheet** to view the data.

Run both cells below in order.

In [ ]:
import warnings
from pathlib import Path

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

warnings.filterwarnings("ignore", message="column count mismatch")

BASE_DIR = Path.cwd()
SAS_DIR = BASE_DIR / "hosp10-sas"
EXTRACTED_DIR = BASE_DIR / "hosp10-extracted"

# Load column mapping (enriched with crosswalk data)
col_map = pd.read_csv(EXTRACTED_DIR / "column_mapping.csv", keep_default_na=False)

# Discover available years
sas_files = sorted(SAS_DIR.glob("prds_hosp10_yr*.sas7bdat"))
available_years = [int(f.stem.split("yr")[-1]) for f in sas_files]

# Build worksheet options grouped by category
ws_data = col_map[col_map["worksheet_name"] != ""].groupby("worksheet_name").agg(
    count=("sas_column", "size"),
    category=("category", "first"),
).sort_index()
ws_options = [(f"{name}  ({row['count']} cols)", name) for name, row in ws_data.iterrows()]

# Global state
data = {"df": None, "year": None, "hospitals": {}}

# Summary stats
has_real_desc = col_map["description"].apply(
    lambda d: bool(d.strip()) and "| Line" not in d
).sum()
print(f"Column mapping: {len(col_map)} columns ({has_real_desc} with descriptions) | "
      f"Years: {available_years[0]}-{available_years[-1]} | Worksheets: {len(ws_data)}")

In [ ]:
# ── Widgets ──────────────────────────────────────────────────────────────────
year_dropdown = widgets.Dropdown(options=available_years, value=available_years[-1], description="Year:")
load_btn = widgets.Button(description="Load Year", button_style="primary")

# Text filter + Dropdown for hospital selection
hosp_filter = widgets.Text(
    value="", placeholder="Type to filter hospitals...",
    description="Filter:", layout=widgets.Layout(width="600px"),
)
hospital_dropdown = widgets.Dropdown(
    options=[], description="Hospital:",
    layout=widgets.Layout(width="600px"),
)

worksheet_dropdown = widgets.Dropdown(
    options=ws_options, description="Worksheet:",
    layout=widgets.Layout(width="600px"),
)

show_all = widgets.Checkbox(value=False, description="Show empty/zero values")
view_btn = widgets.Button(description="View Data", button_style="success")

status_output = widgets.Output()
results_output = widgets.Output()


# ── Filter hospitals as you type ─────────────────────────────────────────────
def update_hospital_filter(change):
    query = hosp_filter.value.strip().upper()
    all_labels = sorted(data["hospitals"].keys())
    if query:
        filtered = [h for h in all_labels if query in h.upper()]
    else:
        filtered = all_labels
    hospital_dropdown.options = filtered
    if filtered:
        hospital_dropdown.value = filtered[0]

hosp_filter.observe(update_hospital_filter, names="value")


# ── Load year ────────────────────────────────────────────────────────────────
def load_year(year):
    path = SAS_DIR / f"prds_hosp10_yr{year}.sas7bdat"
    with status_output:
        clear_output()
        print(f"Loading {path.name}...", end=" ", flush=True)

    df = pd.read_sas(str(path), format="sas7bdat", encoding="latin1")
    data["df"] = df
    data["year"] = year

    # Build hospital lookup using S2_1_C2_2 for state abbreviation
    hospitals = {}
    for _, row in df.iterrows():
        pnum = str(row.get("prvdr_num", "")).strip()
        name = str(row.get("S2_1_C1_3", "")).strip() if "S2_1_C1_3" in df.columns else ""
        if name == "nan":
            name = ""
        st = ""
        if "S2_1_C2_2" in df.columns:
            st = str(row.get("S2_1_C2_2", "")).strip()
            if st == "nan":
                st = ""
        label = f"{pnum} - {name}" + (f" ({st})" if st else "")
        hospitals[label] = pnum
    data["hospitals"] = hospitals

    # Populate dropdown with all hospitals
    hosp_filter.value = ""
    hosp_labels = sorted(hospitals.keys())
    hospital_dropdown.options = hosp_labels
    if hosp_labels:
        hospital_dropdown.value = hosp_labels[0]

    with status_output:
        clear_output()
        print(f"Loaded {year}: {len(df):,} hospitals, {df.shape[1]:,} columns. "
              f"Select a hospital and worksheet, then click View Data.")


def on_load_click(b):
    load_year(year_dropdown.value)

load_btn.on_click(on_load_click)


# ── Helper: parse line_num for proper numeric sorting ────────────────────────
def line_sort_key(val):
    """Convert line_num like '00100', '06250', '200' to a numeric sort key."""
    s = str(val).strip()
    if not s:
        return (0,)
    try:
        return (int(s),)
    except ValueError:
        return (0,)


def col_sort_key(val):
    """Convert column_num like '00100', '00200' to a numeric sort key."""
    s = str(val).strip()
    if not s:
        return (0,)
    try:
        return (int(s),)
    except ValueError:
        return (0,)


# ── View results ─────────────────────────────────────────────────────────────
def on_view_click(b):
    with results_output:
        clear_output()
        df = data["df"]
        if df is None:
            print("No data loaded. Click 'Load Year' first.")
            return

        hosp_label = hospital_dropdown.value
        prvdr_num = data["hospitals"].get(hosp_label)
        if not prvdr_num:
            print(f"Hospital not found: {hosp_label}")
            return

        mask = df["prvdr_num"].astype(str).str.strip() == prvdr_num
        hosp_rows = df[mask]
        if hosp_rows.empty:
            print(f"No data for provider {prvdr_num}")
            return

        row = hosp_rows.iloc[0]
        ws_name = worksheet_dropdown.value
        ws_cols = col_map[col_map["worksheet_name"] == ws_name]

        # Get worksheet category for the header
        ws_category = ""
        ws_subcategory = ""
        if len(ws_cols) > 0:
            ws_category = str(ws_cols.iloc[0].get("category", "")).strip()
            ws_subcategory = str(ws_cols.iloc[0].get("subcategory", "")).strip()
            if ws_category == "nan":
                ws_category = ""
            if ws_subcategory in ("nan", ws_category, ""):
                ws_subcategory = ""

        results = []
        for _, m in ws_cols.iterrows():
            col_name = m["sas_column"]
            if col_name not in df.columns:
                continue
            val = row.get(col_name)

            if not show_all.value:
                if pd.isna(val):
                    continue
                if isinstance(val, (int, float)) and val == 0:
                    continue
                if isinstance(val, str) and val.strip() == "":
                    continue

            # Build the display row
            col_header = str(m.get("column_header", "")).strip()
            if col_header == "nan":
                col_header = ""

            results.append({
                "Description": m["description"],
                "Column Header": col_header,
                "Value": val,
                "Line": m["line_num"],
                "Col #": m["column_num"],
                "SAS Column": col_name,
            })

        # Print header
        print(f"Year: {data['year']}  |  Hospital: {hosp_label}")
        print(f"Worksheet: {ws_name}")
        if ws_category:
            cat_display = ws_category
            if ws_subcategory:
                cat_display += f" > {ws_subcategory}"
            print(f"Category: {cat_display}")
        print(f"{len(results)} fields with data")
        print("-" * 110)

        if results:
            result_df = pd.DataFrame(results)

            # Sort by Line (numeric), then by Col # (numeric)
            result_df["_line_key"] = result_df["Line"].apply(line_sort_key)
            result_df["_col_key"] = result_df["Col #"].apply(col_sort_key)
            result_df = result_df.sort_values(["_line_key", "_col_key"]).drop(
                columns=["_line_key", "_col_key"]
            ).reset_index(drop=True)

            # Format numeric values nicely
            def fmt_val(v):
                if isinstance(v, float) and v == int(v) and abs(v) < 1e15:
                    return f"{int(v):,}"
                if isinstance(v, float):
                    return f"{v:,.2f}"
                return v
            result_df["Value"] = result_df["Value"].apply(fmt_val)
            display(result_df)
        else:
            print("No data found. Try enabling 'Show empty/zero values'.")

view_btn.on_click(on_view_click)


# ── Layout ───────────────────────────────────────────────────────────────────
display(widgets.VBox([
    widgets.HBox([year_dropdown, load_btn]),
    status_output,
    hosp_filter,
    hospital_dropdown,
    worksheet_dropdown,
    widgets.HBox([show_all, view_btn]),
    results_output,
]))

# Auto-load the latest year
load_year(available_years[-1])